# Evidence convergence and binning sensitivity

This notebook holds each simulated periodogram fixed while changing the number of prior draws and the bin width. This is a paired experiment: differences cannot be blamed on a different stochastic realization of the star.

In [1]:
import numpy as np
from asterodetect import (
    AsteroScaleSamples, AstrophysicalInjectionFactory, ObservationModel,
    build_detection_study, run_sensitivity_study,
)
from asterodetect.asteroscale import ASTERO_SCALE_PARAMETERS

In [2]:
rng = np.random.default_rng(123)
latent = rng.normal(size=512)
values = {name: np.ones(512) for name in ASTERO_SCALE_PARAMETERS}
values.update(
    numax=3100 * np.exp(0.025 * latent), dnu=135.1 * np.exp(0.018 * latent),
    FWHM_env=950 * np.exp(0.08 * latent), A_env=2.1 * np.exp(0.12 * latent),
    A_gran=55 * np.exp(-0.10 * latent),
    b_gran_low=760 * np.exp(0.025 * latent),
    b_gran_high=2850 * np.exp(0.025 * latent),
)
samples = AsteroScaleSamples(values)

We generate one noise case, one granulation case, and two oscillation cases. The latter use suppressed and expected amplitudes. For a scientific run, increase both the number of injected realizations and the inference repeats.

In [3]:
factory = AstrophysicalInjectionFactory(samples, duration_days=27.4, cadence_seconds=120)
cases = build_detection_study(
    {'white_noise': [0.1], 'duration_days': [27.4], 'dilution': [1.0]},
    factory, oscillation_amplitudes=[0.3, 1.0], repeats=1, seed=10,
)
[(case.truth, case.metadata['amplitude_scale']) for case in cases]

[('noise', 0.0),
 ('granulation', 0.0),
 ('oscillation', 0.3),
 ('oscillation', 1.0)]

In [4]:
study = run_sensitivity_study(
    cases,
    draw_counts=[32, 128],
    dnu_scales=[0.5, 1.0, 2.0],
    repeats=2,
    seed=11,
    observation=ObservationModel(integration_time_seconds=120),
)
study.summaries()

(SensitivitySummary(draws=32, dnu_scale=0.5, evaluations=8, mean_oscillation_probability=0.3750000883381201, oscillation_probability_std=0.17677654709877266, classification_accuracy=0.875, minimum_median_ess_fraction=0.03125, maximum_median_log_evidence_standard_error=0.9842509842514764),
 SensitivitySummary(draws=32, dnu_scale=1.0, evaluations=8, mean_oscillation_probability=0.3749778494123713, oscillation_probability_std=0.17680859128146986, classification_accuracy=0.875, minimum_median_ess_fraction=0.03125, maximum_median_log_evidence_standard_error=0.9842509842514764),
 SensitivitySummary(draws=32, dnu_scale=2.0, evaluations=8, mean_oscillation_probability=0.12516148752124598, oscillation_probability_std=0.17698830361156204, classification_accuracy=0.625, minimum_median_ess_fraction=0.03125, maximum_median_log_evidence_standard_error=0.9842509842514764),
 SensitivitySummary(draws=128, dnu_scale=0.5, evaluations=8, mean_oscillation_probability=0.2500057077297275, oscillation_probabi

Interpret the diagnostics together. The oscillation-probability scatter should shrink with more draws; the ESS fraction should not collapse; and the log-evidence standard error should decrease. Comparing classification accuracy alone can hide an unstable evidence estimate. The three bin scales test the trade-off between averaging over the mode comb and retaining the broad envelope shape.

In [5]:
rows = [
    (s.draws, s.dnu_scale, s.oscillation_probability_std,
     s.minimum_median_ess_fraction,
     s.maximum_median_log_evidence_standard_error,
     s.classification_accuracy)
    for s in study.summaries()
]
rows

[(32, 0.5, 0.17677654709877266, 0.03125, 0.9842509842514764, 0.875),
 (32, 1.0, 0.17680859128146986, 0.03125, 0.9842509842514764, 0.875),
 (32, 2.0, 0.17698830361156204, 0.03125, 0.9842509842514764, 0.625),
 (128, 0.5, 1.0291357824534424e-05, 0.0078125, 0.9960860906568267, 0.75),
 (128, 1.0, 0.21677166852584784, 0.0078125, 0.9960860906568267, 0.875),
 (128, 2.0, 0.002158668988911116, 0.0078125, 0.9960860906568267, 1.0)]